In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import os
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, STEEL

In [ ]:
# hide
# autorun
SR = 44100
REF_HZ = 1000.0                          # the calibration (and test) tone
REF_START, REF_MIN, REF_MAX = -20.0, -40.0, -6.0    # reference level (dBFS)
DB_STEP = 3.0                            # one press of -3 dB / +3 dB
DB_FLOOR = -100.0                        # assume nobody hears a tone this quiet
DB_TOLERANCE = 2.0                       # stop once the interval is narrower (dB)
DB_SILENT = 20 * np.log10(0.5 / 32767)   # about -96 dBFS: below this, 16-bit WAV is all zeros
MAX_TRIALS = 8                           # room on the x axis


def tone(level_db: float, freq: float, sr: int = SR) -> np.ndarray:
    """One second of sine at an exact level: peak amplitude 10 ** (dBFS / 20)."""
    t = np.arange(sr) / sr
    x = 10 ** (level_db / 20) * np.sin(2 * np.pi * freq * t)
    n = int(0.02 * sr)                   # 20 ms raised-cosine fades: no clicks
    fade = 0.5 - 0.5 * np.cos(np.pi * np.arange(n) / n)
    x[:n] *= fade
    x[-n:] *= fade[::-1]
    return x


def audio_output(x: np.ndarray, sr: int = SR) -> dict:
    """The audio card for x as an output record, at x's own level (no normalizing)."""
    # The card stores 16-bit samples by truncating toward zero, which would
    # silence anything under one step (-90 dBFS). Round to the nearest step
    # instead, as a real converter does (the +-0.5 survives the truncation).
    q = np.round(x * 32767)
    x = (q + 0.5 * np.sign(q)) / 32767
    audio = Audio(x, rate=sr, normalize=False)
    data, metadata = get_ipython().display_formatter.format(audio)
    return {"output_type": "display_data", "data": data, "metadata": metadata}


class Bisection:
    """Binary search for a listener's threshold between a value they hear and one they don't."""

    def __init__(self, heard: float, missed: float, tolerance: float):
        self.heard = heard               # closest value known to be audible
        self.missed = missed             # closest value known to be inaudible
        self.tolerance = tolerance
        self.trials: list[tuple[float, bool]] = []

    @property
    def probe(self) -> float:
        """The value to test next: the middle of the remaining interval."""
        return (self.heard + self.missed) / 2

    @property
    def done(self) -> bool:
        return abs(self.heard - self.missed) < self.tolerance

    def answer(self, heard: bool) -> None:
        probe = self.probe
        self.trials.append((probe, heard))
        if heard:
            self.heard = probe
        else:
            self.missed = probe


def dbfs(level: float) -> str:
    return f"{level:.0f} dBFS".replace("-", "−")


def figure():
    fig = go.Figure()
    # 0: the remaining interval, a gold band (empty until the test starts)
    fig.add_scatter(x=[], y=[], mode="lines", fill="toself", fillcolor=GOLD,
                    opacity=0.3, line=dict(width=0), name="remaining interval",
                    hoverinfo="skip")
    # 1: the reference level
    fig.add_scatter(x=[0, MAX_TRIALS + 0.5], y=[REF_START] * 2, mode="lines+text",
                    line=dict(color=STEEL, width=2, dash="dash"), text=["", "reference"],
                    textposition="top left", textfont=dict(color=IRON), cliponaxis=False,
                    name="reference", showlegend=False, hoverinfo="skip")
    # 2, 3: the levels tested so far
    fig.add_scatter(x=[], y=[], mode="markers", marker=dict(color=BLUE, size=10),
                    name="heard", hovertemplate="trial %{x}: %{y:.1f} dBFS<extra>heard</extra>")
    fig.add_scatter(x=[], y=[], mode="markers", marker=dict(color=RED, size=10),
                    name="not heard",
                    hovertemplate="trial %{x}: %{y:.1f} dBFS<extra>not heard</extra>")
    # 4: the level on the audio card now
    fig.add_scatter(x=[], y=[], mode="markers",
                    marker=dict(color="white", size=10, line=dict(color=GOLD, width=3)),
                    name="next tone", hovertemplate="trial %{x}: %{y:.1f} dBFS<extra>next</extra>")
    fig.update_xaxes(range=[0, MAX_TRIALS + 0.5], dtick=1, title_text="Trial",
                     fixedrange=True)
    fig.update_yaxes(range=[-105, 0], title_text="Level (dBFS)", fixedrange=True)
    fig.update_layout(showlegend=True)
    return fig


def controls(fig):
    state = {"ref": REF_START, "search": None}
    text = widgets.HTML()
    down = widgets.Button(description="−3 dB")
    up = widgets.Button(description="+3 dB")
    start = widgets.Button(description="Start test", button_style="primary")
    heard = widgets.Button(description="I heard it")
    missed = widgets.Button(description="I didn't hear it")
    restart = widgets.Button(description="Restart", icon="refresh")
    for button, color in ((heard, BLUE), (missed, RED)):
        button.style.button_color = color
        button.style.text_color = "white"
    row = widgets.HBox()
    out = widgets.Output()

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def redraw(MAX_TRIALS=MAX_TRIALS):
        search = state["search"]
        with fig.batch_update():
            fig.data[1].y = [state["ref"]] * 2
            if search is None:
                for k in (0, 2, 3, 4):
                    fig.data[k].x, fig.data[k].y = [], []
                return
            right = MAX_TRIALS + 0.5
            lo, hi = sorted((search.heard, search.missed))
            fig.data[0].x = [0, right, right, 0, 0]
            fig.data[0].y = [lo, lo, hi, hi, lo]
            for k, answer in ((2, True), (3, False)):
                fig.data[k].x = [i + 1 for i, (_, a) in enumerate(search.trials) if a == answer]
                fig.data[k].y = [v for v, a in search.trials if a == answer]
            n = len(search.trials)
            fig.data[4].x, fig.data[4].y = ([], []) if search.done else ([n + 1], [search.probe])

    def render(tone=tone, audio_output=audio_output, REF_HZ=REF_HZ):
        search = state["search"]
        level = state["ref"] if search is None else search.probe
        out.outputs = (audio_output(tone(level, REF_HZ)),)

    def paint(dbfs=dbfs, REF_MIN=REF_MIN, REF_MAX=REF_MAX, DB_SILENT=DB_SILENT):
        search = state["search"]
        ref = state["ref"]
        if search is None:
            text.value = (
                "<b>Step 1: calibrate.</b> Put in headphones and play the tone. Use the "
                "buttons until it is loud but comfortable, then press <b>Start test</b>. "
                f"Reference: <b>{dbfs(ref)}</b>.")
            down.disabled = ref <= REF_MIN
            up.disabled = ref >= REF_MAX
            row.children = (down, up, start)
        elif not search.done:
            text.value = (
                f"<b>Step 2, trial {len(search.trials) + 1}.</b> Play the tone (it may be very "
                "quiet, so listen closely). Did you hear it?")
            row.children = (heard, missed)
        else:
            threshold = search.probe
            text.value = (
                f"<b>Result.</b> The quietest level you can hear is about <b>{dbfs(threshold)}</b>, "
                f"so your effective dynamic range is <b>{ref - threshold:.0f} dB</b> "
                f"({dbfs(ref)} down to {dbfs(threshold)}). The card holds a tone at that level.")
            if threshold < DB_SILENT + 6:
                text.value += (
                    " Careful: the card stores 16-bit samples, so anything below about "
                    "−96 dBFS is digital silence (all zeros). A result this close to that "
                    "floor says more about the 16-bit format than about your ears.")
            row.children = (restart,)

    def changed():
        paint()
        redraw()
        render()

    def nudge(step, REF_MIN=REF_MIN, REF_MAX=REF_MAX):
        state["ref"] = float(np.clip(state["ref"] + step, REF_MIN, REF_MAX))
        changed()

    def begin(_, Bisection=Bisection, DB_FLOOR=DB_FLOOR, DB_TOLERANCE=DB_TOLERANCE):
        state["search"] = Bisection(state["ref"], DB_FLOOR, DB_TOLERANCE)
        changed()

    def respond(answer):
        state["search"].answer(answer)
        changed()

    def reset(_):
        state["search"] = None           # keep the calibrated reference level
        changed()

    down.on_click(lambda _, step=DB_STEP: nudge(-step))
    up.on_click(lambda _, step=DB_STEP: nudge(step))
    start.on_click(begin)
    heard.on_click(lambda _: respond(True))
    missed.on_click(lambda _: respond(False))
    restart.on_click(reset)
    paint()
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([text, row, out])


icm_plotly.show(figure, controls)